# Data Provenance and Reproducibility Audit

## Purpose

This notebook documents the provenance, integrity, reproducibility and version consistency of the Mnjoli W60E hydroclimatic dataset.

The original modelling dataset and independent test results must remain unchanged during this audit.

## 1. Audit Environment and Dataset Inventory

Record source files, dimensions, date coverage, modification timestamps and file hashes.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import json
from datetime import datetime

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

print('Project root:', PROJECT_ROOT.resolve())
print('Audit initiated:', datetime.now().isoformat())

Project root: C:\Users\Para\Desktop\Eswatini_Water_Stress_ML
Audit initiated: 2026-11-05T00:23:28.554092


In [3]:
# ============================================================
# 2. DATASET INVENTORY
# ============================================================

files_to_audit = [
    "data/external/mnjoli_era5_land_daily_2015_2025_raw.csv",
    "data/external/mnjoli_era5_land_daily_2015_2025_processed.csv",
    "data/processed/mnjoli_water_stress_monthly.csv",
    "data/processed/mnjoli_water_stress_ml_dataset.csv",
    "data/processed/mnjoli_water_stress_modeling_dataset.csv",
    "outputs/independent_test_performance.csv",
    "outputs/independent_test_forecasts.csv"
]

inventory = []

for relative_path in files_to_audit:
    file_path = PROJECT_ROOT / relative_path

    if not file_path.exists():
        inventory.append({
            "File": relative_path,
            "Exists": False
        })
        continue

    stat = file_path.stat()

    inventory.append({
        "File": relative_path,
        "Exists": True,
        "Size_MB": round(stat.st_size / (1024 ** 2), 3),
        "Modified": pd.Timestamp(stat.st_mtime, unit="s"),
        "SHA256": hashlib.sha256(
            file_path.read_bytes()
        ).hexdigest()
    })

inventory_df = pd.DataFrame(inventory)

display(inventory_df)

,File,Exists,Size_MB,Modified,SHA256
0,data/external/mnjoli_era5_land_daily_2015_2025...,True,1.009,2026-11-04 07:57:44.030216217,bfdb2ad19cc5212cf798f35cb409826fd264dd6aac7a36...
1,data/external/mnjoli_era5_land_daily_2015_2025...,True,1.159,2026-11-04 07:57:44.386424303,ca2f14f004f0235ea0fed5729f5e7f6e4f9b6a7eb1e4a0...
2,data/processed/mnjoli_water_stress_monthly.csv,True,0.042,2026-11-04 08:00:04.416238308,7b29c1d800b6fdc3fd7e0ae0d328663c0ed7bc3f5bcbbc...
3,data/processed/mnjoli_water_stress_ml_dataset.csv,True,0.136,2026-11-04 08:00:13.247523069,ff65d1f54343fb6706fe72cb5db76637449f3b84ee4713...
4,data/processed/mnjoli_water_stress_modeling_da...,True,0.136,2026-08-10 23:16:39.534451962,29dbddd983670b0572471aeb815776d5e0130343c27a18...
5,outputs/independent_test_performance.csv,True,0.000,2026-11-04 08:35:02.211476564,02595ec0c6f6997407eb6a55aaaaccfd48bc01764eb3eb...
6,outputs/independent_test_forecasts.csv,True,0.002,2026-11-04 08:35:02.217375517,b47128f34f16d055ceb186c19e5a89b25c48b82659bfbd...


In [4]:
# ============================================================
# 3. DATASET STRUCTURE AND INTEGRITY
# ============================================================

dataset_files = [
    "data/external/mnjoli_era5_land_daily_2015_2025_raw.csv",
    "data/external/mnjoli_era5_land_daily_2015_2025_processed.csv",
    "data/processed/mnjoli_water_stress_monthly.csv",
    "data/processed/mnjoli_water_stress_ml_dataset.csv",
    "data/processed/mnjoli_water_stress_modeling_dataset.csv",
    "outputs/independent_test_performance.csv",
    "outputs/independent_test_forecasts.csv"
]

date_candidates = [
    "date",
    "month_date",
    "forecast_date"
]

structure_results = []
date_results = []

for relative_path in dataset_files:

    file_path = PROJECT_ROOT / relative_path

    if not file_path.exists():
        continue

    data = pd.read_csv(file_path)

    structure_results.append({
        "File": relative_path,
        "Rows": len(data),
        "Columns": len(data.columns),
        "Missing_Cells": int(data.isna().sum().sum()),
        "Duplicate_Rows": int(data.duplicated().sum())
    })

    for column in date_candidates:

        if column not in data.columns:
            continue

        dates = pd.to_datetime(
            data[column],
            errors="coerce"
        )

        date_results.append({
            "File": relative_path,
            "Date_Column": column,
            "Valid_Dates": int(dates.notna().sum()),
            "Invalid_Dates": int(dates.isna().sum()),
            "Start_Date": dates.min(),
            "End_Date": dates.max(),
            "Duplicate_Dates": int(dates.duplicated().sum())
        })

structure_df = pd.DataFrame(structure_results)
date_df = pd.DataFrame(date_results)

print("DATASET STRUCTURE")
display(structure_df)

print("\nDATE COVERAGE AND ALIGNMENT")
display(date_df)

DATASET STRUCTURE


,File,Rows,Columns,Missing_Cells,Duplicate_Rows
0,data/external/mnjoli_era5_land_daily_2015_2025...,4017,14,0,0
1,data/external/mnjoli_era5_land_daily_2015_2025...,4017,16,0,0
2,data/processed/mnjoli_water_stress_monthly.csv,132,20,7,0
3,data/processed/mnjoli_water_stress_ml_dataset.csv,126,63,0,0
4,data/processed/mnjoli_water_stress_modeling_da...,126,62,0,0
5,outputs/independent_test_performance.csv,3,4,0,0
6,outputs/independent_test_forecasts.csv,24,5,0,0



DATE COVERAGE AND ALIGNMENT


,File,Date_Column,Valid_Dates,Invalid_Dates,Start_Date,End_Date,Duplicate_Dates
0,data/external/mnjoli_era5_land_daily_2015_2025...,date,4017,0,2015-01-01,2025-12-30,0
1,data/external/mnjoli_era5_land_daily_2015_2025...,date,4017,0,2015-01-01,2025-12-30,0
2,data/processed/mnjoli_water_stress_monthly.csv,month_date,132,0,2015-01-01,2025-12-01,0
3,data/processed/mnjoli_water_stress_ml_dataset.csv,month_date,126,0,2015-07-01,2025-12-01,0
4,data/processed/mnjoli_water_stress_modeling_da...,month_date,126,0,2015-06-01,2025-11-01,0
5,data/processed/mnjoli_water_stress_modeling_da...,forecast_date,126,0,2015-07-01,2025-12-01,0
6,outputs/independent_test_forecasts.csv,forecast_date,24,0,2024-01-01,2025-12-01,0


In [5]:
# ============================================================
# 4. DAILY DATE CONTINUITY AUDIT
# ============================================================

daily_files = [
    "data/external/mnjoli_era5_land_daily_2015_2025_raw.csv",
    "data/external/mnjoli_era5_land_daily_2015_2025_processed.csv"
]

for relative_path in daily_files:

    data = pd.read_csv(
        PROJECT_ROOT / relative_path,
        parse_dates=["date"]
    )

    actual_dates = pd.DatetimeIndex(
        data["date"].dropna().unique()
    ).sort_values()

    expected_dates = pd.date_range(
        start="2015-01-01",
        end="2025-12-31",
        freq="D"
    )

    missing_dates = expected_dates.difference(
        actual_dates
    )

    print("\n" + "=" * 65)
    print(relative_path)
    print("=" * 65)

    print("Actual daily records:", len(actual_dates))
    print("Expected calendar days:", len(expected_dates))
    print("Missing dates:", len(missing_dates))

    print("\nMissing dates:")
    print(missing_dates.tolist())

    print("\nFirst date:", actual_dates.min())
    print("Last date:", actual_dates.max())


data/external/mnjoli_era5_land_daily_2015_2025_raw.csv
Actual daily records: 4017
Expected calendar days: 4018
Missing dates: 1

Missing dates:
[Timestamp('2025-12-31 00:00:00')]

First date: 2015-01-01 00:00:00
Last date: 2025-12-30 00:00:00

data/external/mnjoli_era5_land_daily_2015_2025_processed.csv
Actual daily records: 4017
Expected calendar days: 4018
Missing dates: 1

Missing dates:
[Timestamp('2025-12-31 00:00:00')]

First date: 2015-01-01 00:00:00
Last date: 2025-12-30 00:00:00


In [6]:
# ============================================================
# 5. CURRENT ERA5 DATA VS COMMITTED GIT VERSION
# ============================================================

import io
import subprocess
import numpy as np

era5_files = [
    "data/external/mnjoli_era5_land_daily_2015_2025_raw.csv",
    "data/external/mnjoli_era5_land_daily_2015_2025_processed.csv"
]

comparison_results = []

git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

print("Current Git commit:", git_commit)

for relative_path in era5_files:

    current_df = pd.read_csv(
        PROJECT_ROOT / relative_path
    )

    committed_bytes = subprocess.run(
        ["git", "show", f"HEAD:{relative_path}"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        check=True
    ).stdout

    committed_df = pd.read_csv(
        io.BytesIO(committed_bytes)
    )

    print("\n" + "=" * 70)
    print(relative_path)
    print("=" * 70)

    print("Current shape:", current_df.shape)
    print("Committed shape:", committed_df.shape)

    if not current_df["date"].equals(committed_df["date"]):
        raise ValueError(
            f"Date alignment differs in {relative_path}"
        )

    common_columns = sorted(
        set(current_df.columns) & set(committed_df.columns)
    )

    for column in common_columns:

        if column == "date":
            continue

        current_values = current_df[column]
        committed_values = committed_df[column]

        if (
            pd.api.types.is_numeric_dtype(current_values)
            and pd.api.types.is_numeric_dtype(committed_values)
        ):

            difference = (
                current_values - committed_values
            ).abs()

            changed = difference > 1e-12

            comparison_results.append({
                "File": relative_path,
                "Variable": column,
                "Changed_Rows": int(changed.sum()),
                "Maximum_Absolute_Difference": float(
                    difference.max()
                )
            })

        else:

            changed = (
                current_values.fillna("<NA>").astype(str)
                != committed_values.fillna("<NA>").astype(str)
            )

            comparison_results.append({
                "File": relative_path,
                "Variable": column,
                "Changed_Rows": int(changed.sum()),
                "Maximum_Absolute_Difference": "Non-numeric"
            })

comparison_df = pd.DataFrame(comparison_results)

display(
    comparison_df[
        comparison_df["Changed_Rows"] > 0
    ].sort_values(
        ["File", "Changed_Rows"],
        ascending=[True, False]
    )
)

Current Git commit: 6a9b0e903e311fdd398588482871a615a8f50343

data/external/mnjoli_era5_land_daily_2015_2025_raw.csv
Current shape: (4017, 14)
Committed shape: (4017, 14)

data/external/mnjoli_era5_land_daily_2015_2025_processed.csv
Current shape: (4017, 16)
Committed shape: (4017, 16)


,File,Variable,Changed_Rows,Maximum_Absolute_Difference
14,data/external/mnjoli_era5_land_daily_2015_2025...,pet_m,2711,0.000319
15,data/external/mnjoli_era5_land_daily_2015_2025...,pet_mm,2711,0.318995
16,data/external/mnjoli_era5_land_daily_2015_2025...,potential_evaporation_sum,2711,0.000319
17,data/external/mnjoli_era5_land_daily_2015_2025...,runoff_sum,2435,0.002263
19,data/external/mnjoli_era5_land_daily_2015_2025...,surface_runoff_sum,1739,0.002263
20,data/external/mnjoli_era5_land_daily_2015_2025...,surface_solar_radiation_downwards_sum,125,2.613718
1,data/external/mnjoli_era5_land_daily_2015_2025...,potential_evaporation_sum,2711,0.000319
2,data/external/mnjoli_era5_land_daily_2015_2025...,runoff_sum,2435,0.002263
4,data/external/mnjoli_era5_land_daily_2015_2025...,surface_runoff_sum,1739,0.002263
5,data/external/mnjoli_era5_land_daily_2015_2025...,surface_solar_radiation_downwards_sum,125,2.613718


In [7]:
# ============================================================
# 6. CURRENT MONTHLY DATA VS COMMITTED GIT VERSION
# ============================================================

relative_path = (
    "data/processed/mnjoli_water_stress_monthly.csv"
)

current_monthly = pd.read_csv(
    PROJECT_ROOT / relative_path
)

committed_bytes = subprocess.run(
    ["git", "show", f"HEAD:{relative_path}"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    check=True
).stdout

committed_monthly = pd.read_csv(
    io.BytesIO(committed_bytes)
)

print("Current monthly shape:", current_monthly.shape)
print("Committed monthly shape:", committed_monthly.shape)

if not current_monthly["month_date"].equals(
    committed_monthly["month_date"]
):
    raise ValueError("Monthly dates do not align.")

monthly_comparison = []

common_columns = sorted(
    set(current_monthly.columns)
    & set(committed_monthly.columns)
)

for column in common_columns:

    if column == "month_date":
        continue

    current_values = current_monthly[column]
    committed_values = committed_monthly[column]

    if (
        pd.api.types.is_numeric_dtype(current_values)
        and pd.api.types.is_numeric_dtype(committed_values)
    ):

        difference = (
            current_values - committed_values
        ).abs()

        changed = difference > 1e-12

        monthly_comparison.append({
            "Variable": column,
            "Changed_Months": int(changed.sum()),
            "Maximum_Absolute_Difference": float(
                difference.max()
            )
        })

monthly_comparison_df = pd.DataFrame(
    monthly_comparison
)

display(
    monthly_comparison_df[
        monthly_comparison_df["Changed_Months"] > 0
    ].sort_values(
        "Changed_Months",
        ascending=False
    )
)

Current monthly shape: (132, 20)
Committed monthly shape: (132, 14)


,Variable,Changed_Months,Maximum_Absolute_Difference
1,pet_mm,103,0.244695
3,runoff_mm,97,0.000016
8,surface_runoff_mm,82,0.000016
6,solar_radiation,7,0.087124


In [8]:
# ============================================================
# 7. MONTHLY DATASET SCHEMA COMPARISON
# ============================================================

current_columns = set(current_monthly.columns)
committed_columns = set(committed_monthly.columns)

print("Columns in both versions:")
print(sorted(current_columns & committed_columns))

print("\nColumns only in current monthly dataset:")
print(sorted(current_columns - committed_columns))

print("\nColumns only in committed monthly dataset:")
print(sorted(committed_columns - current_columns))

print("\nCurrent monthly data types:")
display(
    current_monthly.dtypes.rename("Current dtype").to_frame()
)

print("\nCommitted monthly data types:")
display(
    committed_monthly.dtypes.rename("Committed dtype").to_frame()
)

Columns in both versions:
['dewpoint_c', 'month_date', 'pet_mm', 'precipitation_mm', 'runoff_mm', 'soil_moisture_layer1', 'soil_moisture_layer2', 'solar_radiation', 'surface_pressure_kpa', 'surface_runoff_mm', 'temperature_max_c', 'temperature_mean_c', 'temperature_min_c', 'wind_speed']

Columns only in current monthly dataset:
['month', 'season', 'water_balance_3month', 'water_balance_6month', 'water_balance_mm', 'year']

Columns only in committed monthly dataset:
[]

Current monthly data types:


,Current dtype
month_date,str
precipitation_mm,float64
pet_mm,float64
temperature_mean_c,float64
temperature_min_c,float64
temperature_max_c,float64
dewpoint_c,float64
soil_moisture_layer1,float64
soil_moisture_layer2,float64
runoff_mm,float64



Committed monthly data types:


,Committed dtype
month_date,str
precipitation_mm,float64
pet_mm,float64
temperature_mean_c,float64
temperature_min_c,float64
temperature_max_c,float64
dewpoint_c,float64
soil_moisture_layer1,float64
soil_moisture_layer2,float64
runoff_mm,float64


In [9]:
# ============================================================
# 8. WATER-BALANCE DERIVATION VALIDATION
# ============================================================

monthly_check = current_monthly.copy()

monthly_check["month_date"] = pd.to_datetime(
    monthly_check["month_date"]
)

monthly_check = monthly_check.sort_values(
    "month_date"
).reset_index(drop=True)

# Independently reconstruct water balance
monthly_check["water_balance_reconstructed"] = (
    monthly_check["precipitation_mm"]
    - monthly_check["pet_mm"]
)

# Independently reconstruct rolling accumulations
monthly_check["water_balance_3month_reconstructed"] = (
    monthly_check["water_balance_reconstructed"]
    .rolling(window=3, min_periods=3)
    .sum()
)

monthly_check["water_balance_6month_reconstructed"] = (
    monthly_check["water_balance_reconstructed"]
    .rolling(window=6, min_periods=6)
    .sum()
)

checks = {
    "water_balance_mm": "water_balance_reconstructed",
    "water_balance_3month": "water_balance_3month_reconstructed",
    "water_balance_6month": "water_balance_6month_reconstructed"
}

validation_results = []

for original, reconstructed in checks.items():

    difference = (
        monthly_check[original]
        - monthly_check[reconstructed]
    ).abs()

    validation_results.append({
        "Variable": original,
        "Compared_Rows": int(difference.notna().sum()),
        "Maximum_Absolute_Difference": float(
            difference.max()
        ),
        "Mismatched_Rows": int(
            (difference > 1e-10).sum()
        )
    })

validation_df = pd.DataFrame(validation_results)

display(validation_df)

,Variable,Compared_Rows,Maximum_Absolute_Difference,Mismatched_Rows
0,water_balance_mm,132,8.526513e-14,0
1,water_balance_3month,130,1.705303e-13,0
2,water_balance_6month,127,2.273737e-13,0


In [10]:
# ============================================================
# 9. INDEPENDENT WSI RECONSTRUCTION AND TARGET COMPARISON
# ============================================================

monthly = current_monthly.copy()

monthly["month_date"] = pd.to_datetime(
    monthly["month_date"]
)

monthly = monthly.sort_values(
    "month_date"
).reset_index(drop=True)

# Training-period climatology only: Jan 2015–Dec 2021
training_period = monthly[
    monthly["month_date"] <= "2021-12-01"
].copy()

training_climatology = (
    training_period
    .groupby(training_period["month_date"].dt.month)[
        "water_balance_3month"
    ]
    .agg(["mean", "std"])
)

# Reconstruct WSI using training-period calendar-month statistics
monthly["calendar_month"] = monthly["month_date"].dt.month

monthly = monthly.join(
    training_climatology,
    on="calendar_month"
)

monthly["wsi_reconstructed"] = (
    monthly["water_balance_3month"] - monthly["mean"]
) / monthly["std"]

# Construct the next-month target independently
monthly["forecast_date"] = (
    monthly["month_date"].shift(-1)
)

monthly["target_reconstructed"] = (
    monthly["wsi_reconstructed"].shift(-1)
)

# Load frozen modelling dataset
frozen = pd.read_csv(
    PROJECT_ROOT
    / "data/processed/mnjoli_water_stress_modeling_dataset.csv"
)

frozen["forecast_date"] = pd.to_datetime(
    frozen["forecast_date"]
)

comparison = frozen[
    ["forecast_date", "target_t_plus_1"]
].merge(
    monthly[
        ["forecast_date", "target_reconstructed"]
    ],
    on="forecast_date",
    how="inner",
    validate="one_to_one"
)

difference = (
    comparison["target_t_plus_1"]
    - comparison["target_reconstructed"]
).abs()

print("Frozen target rows:", len(frozen))
print("Matched target rows:", len(comparison))
print("Unmatched target rows:", len(frozen) - len(comparison))
print("Mean absolute difference:", difference.mean())
print("Maximum absolute difference:", difference.max())
print("Exact match within 1e-10:", bool((difference < 1e-10).all()))

display(comparison.head())

Frozen target rows: 126
Matched target rows: 126
Unmatched target rows: 0
Mean absolute difference: 0.0013608757969091722
Maximum absolute difference: 0.00979489955591889
Exact match within 1e-10: False


,forecast_date,target_t_plus_1,target_reconstructed
0,2015-07-01,-0.863509,-0.861666
1,2015-08-01,-0.414403,-0.410563
2,2015-09-01,1.109906,1.110516
3,2015-10-01,-0.955608,-0.955727
4,2015-11-01,-1.400808,-1.400530


In [11]:
# ============================================================
# 10. RECONSTRUCT FROZEN TARGETS FROM COMMITTED MONTHLY DATA
# ============================================================

relative_path = (
    "data/processed/mnjoli_water_stress_monthly.csv"
)

committed_bytes = subprocess.run(
    ["git", "show", f"HEAD:{relative_path}"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    check=True
).stdout

original_monthly = pd.read_csv(
    io.BytesIO(committed_bytes)
)

original_monthly["month_date"] = pd.to_datetime(
    original_monthly["month_date"]
)

original_monthly = original_monthly.sort_values(
    "month_date"
).reset_index(drop=True)

# Reconstruct water balance from committed climate variables
original_monthly["water_balance_mm"] = (
    original_monthly["precipitation_mm"]
    - original_monthly["pet_mm"]
)

original_monthly["water_balance_3month"] = (
    original_monthly["water_balance_mm"]
    .rolling(3, min_periods=3)
    .sum()
)

# Training-only calendar-month climatology
original_training = original_monthly[
    original_monthly["month_date"] <= "2021-12-01"
].copy()

original_training["calendar_month"] = (
    original_training["month_date"].dt.month
)

original_climatology = (
    original_training
    .groupby("calendar_month")["water_balance_3month"]
    .agg(["mean", "std"])
)

original_monthly["calendar_month"] = (
    original_monthly["month_date"].dt.month
)

original_monthly = original_monthly.join(
    original_climatology,
    on="calendar_month"
)

# Reconstruct standardised water stress index
original_monthly["wsi_original"] = (
    original_monthly["water_balance_3month"]
    - original_monthly["mean"]
) / original_monthly["std"]

# Reconstruct one-month-ahead target
original_monthly["forecast_date"] = (
    original_monthly["month_date"].shift(-1)
)

original_monthly["target_original"] = (
    original_monthly["wsi_original"].shift(-1)
)

# Compare with frozen modelling targets
original_comparison = frozen[
    ["forecast_date", "target_t_plus_1"]
].merge(
    original_monthly[
        ["forecast_date", "target_original"]
    ],
    on="forecast_date",
    how="inner",
    validate="one_to_one"
)

original_difference = (
    original_comparison["target_t_plus_1"]
    - original_comparison["target_original"]
).abs()

print("Committed Git version:", git_commit)
print("Frozen target rows:", len(frozen))
print("Matched target rows:", len(original_comparison))
print("Mean absolute difference:", original_difference.mean())
print("Maximum absolute difference:", original_difference.max())
print(
    "Exact match within 1e-10:",
    bool((original_difference < 1e-10).all())
)

display(original_comparison.head())

Committed Git version: 6a9b0e903e311fdd398588482871a615a8f50343
Frozen target rows: 126
Matched target rows: 126
Mean absolute difference: 3.0178681423342546e-17
Maximum absolute difference: 2.220446049250313e-16
Exact match within 1e-10: True


,forecast_date,target_t_plus_1,target_original
0,2015-07-01,-0.863509,-0.863509
1,2015-08-01,-0.414403,-0.414403
2,2015-09-01,1.109906,1.109906
3,2015-10-01,-0.955608,-0.955608
4,2015-11-01,-1.400808,-1.400808


## Verified finding: Frozen target provenance

The 126 one-month-ahead target values in the frozen modelling dataset were independently reconstructed from the monthly climate dataset committed in Git at commit `6a9b0e903e311fdd398588482871a615a8f50343`.

The reconstruction used:
- Monthly water balance = precipitation minus PET.
- Three-month rolling water balance.
- Calendar-month climatology calculated using January 2015–December 2021 only.
- Standardised WSI calculated using the corresponding training-period monthly mean and standard deviation.
- One-month-ahead target alignment using the subsequent month's WSI.

Results:
- Matched target observations: 126/126.
- Mean absolute difference: 3.02 × 10⁻¹⁷.
- Maximum absolute difference: 2.22 × 10⁻¹⁶.
- All targets matched within a tolerance of 10⁻¹⁰.

The current November 2026 monthly dataset does not reproduce the frozen targets exactly (MAE difference: 0.001361; maximum absolute difference: 0.009795). This discrepancy is associated with changes in upstream ERA5 variables between the committed and current data versions.

Conclusion: The frozen target series is reproducible from the committed historical monthly dataset. The original model evaluation must remain associated with that frozen data version. The reason for the upstream ERA5 changes remains unresolved.

In [12]:
import ee

ee.Initialize(project="eswatiniwaterstress")

years = range(1981, 2026)

chirps_asset = "UCSB-CHG/CHIRPS/DAILY"
era5_asset = "ECMWF/ERA5_LAND/DAILY_AGGR"

required_bands = [
    "temperature_2m",
    "temperature_2m_min",
    "temperature_2m_max",
    "dewpoint_temperature_2m",
    "surface_pressure",
    "u_component_of_wind_10m",
    "v_component_of_wind_10m",
    "surface_solar_radiation_downwards_sum",
    "volumetric_soil_water_layer_1",
    "volumetric_soil_water_layer_2",
    "runoff_sum",
    "surface_runoff_sum",
    "potential_evaporation_sum"
]

print("YEAR | CHIRPS DAYS | ERA5 DAYS | REQUIRED BANDS")
print("-" * 65)

for year in years:
    start = f"{year}-01-01"
    end = f"{year + 1}-01-01"

    chirps = (
        ee.ImageCollection(chirps_asset)
        .filterDate(start, end)
    )

    era5 = (
        ee.ImageCollection(era5_asset)
        .filterDate(start, end)
    )

    chirps_count = chirps.size().getInfo()
    era5_count = era5.size().getInfo()

    first = ee.Image(era5.first())
    available = first.bandNames().getInfo()

    missing = [
        band for band in required_bands
        if band not in available
    ]

    print(
        f"{year} | {chirps_count} | {era5_count} | "
        f"{'OK' if not missing else 'MISSING: ' + str(missing)}"
    )

    if chirps_count == 0 or era5_count == 0 or missing:
        print("WARNING: Coverage issue detected.")



c:\Users\Para\Desktop\Eswatini_Water_Stress_ML\.venv\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


YEAR | CHIRPS DAYS | ERA5 DAYS | REQUIRED BANDS
-----------------------------------------------------------------
1981 | 365 | 365 | OK
1982 | 365 | 365 | OK
1983 | 365 | 365 | OK
1984 | 366 | 366 | OK
1985 | 365 | 365 | OK
1986 | 365 | 365 | OK
1987 | 365 | 365 | OK
1988 | 366 | 366 | OK
1989 | 365 | 365 | OK
1990 | 365 | 365 | OK
1991 | 365 | 365 | OK
1992 | 366 | 366 | OK
1993 | 365 | 365 | OK
1994 | 365 | 365 | OK
1995 | 365 | 365 | OK
1996 | 366 | 366 | OK
1997 | 365 | 365 | OK
1998 | 365 | 365 | OK
1999 | 365 | 365 | OK
2000 | 366 | 366 | OK
2001 | 365 | 365 | OK
2002 | 365 | 365 | OK
2003 | 365 | 365 | OK
2004 | 366 | 366 | OK
2005 | 365 | 365 | OK
2006 | 365 | 365 | OK
2007 | 365 | 365 | OK
2008 | 366 | 366 | OK
2009 | 365 | 365 | OK
2010 | 365 | 365 | OK
2011 | 365 | 365 | OK
2012 | 366 | 366 | OK
2013 | 365 | 365 | OK
2014 | 365 | 365 | OK
2015 | 365 | 365 | OK
2016 | 366 | 366 | OK
2017 | 365 | 365 | OK
2018 | 365 | 365 | OK
2019 | 365 | 365 | OK
2020 | 366 | 366 | OK
2021 |

## 2. Source Dataset Integrity

Inspect dimensions, date ranges, duplicate dates, missing values and checksums.

## 3. ERA5-Land Extraction Provenance

Document the Earth Engine collection, temporal filter, catchment geometry, selected bands, spatial reducer, scale and extraction procedure.

## 4. Historical Version Comparison

Compare committed and current daily ERA5-Land values and identify changed variables and dates.

## 5. Variable Transformation Verification

Verify PET, temperature, runoff and other derived variables against their source values.

## 6. Water Stress Index Reconstruction

Reconstruct the training-period climatology and WSI and compare against the frozen modelling dataset.

## 7. Frozen Model Reproducibility

Verify that the original feature specification, training split, test predictions and evaluation metrics remain reproducible.

## 8. Audit Findings and Decisions

Record verified findings, unresolved questions, limitations and decisions before dataset expansion.